# Galea quality report

Interactive data-quality report for one Galea / OpenBCI GUI recording. Every number is measured from the data when the cells run.

* **Data:** a session folder, the folder holding `BrainFlow-RAW_*.csv` and the `OpenBCI-*.txt` files. With `DATA = None` (below), running section 1 opens a folder dialog, starting at the last folder chosen. To skip the dialog, set `DATA` to a path, or set `$GALEA_DATA`.
* **Clock:** ExG (250 Hz) and Aux (50 Hz) share the headset clock. It is mapped to PC time by a linear fit to the lower envelope of the arrival timestamps. `t` = seconds since the first valid ExG sample.
* **Valid range:** the part of the recording you confirm as usable (e.g. after headset setup). The report suggests a range; you confirm or adjust it in the browser (section 5), and it is saved with the dataset in `galea_session.json`. While no range is confirmed, quality metrics use the whole recording.
* Only raw signals are shown here. Derived signals (heart rate, breathing, head motion, EMG envelopes) are in `analysis.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import ipywidgets as W
from IPython.display import Markdown, display
from galea import load_session, resolve_data, qc_report, channel_quality, analysis_range
from galea import qc_plots as qp
from galea.browser import SignalBrowser

DATA = None          # session folder path; None → folder dialog (or $GALEA_DATA if set)
pd.set_option("display.max_colwidth", 400)

## 1. Session

In [ ]:
s = load_session(resolve_data(DATA))
qp.session_summary(s)

## 2. Findings
Automated checks, coloured by severity: red = error, yellow = warning, green = ok. They reflect the valid range as it was when this cell ran.

In [ ]:
q = qc_report(s)
q.style.apply(lambda r: [f"background-color: {qp.SEV_COLORS[r.severity]}"] * len(r), axis=1).hide(axis="index")

## 3. Clock
Offset between PC arrival time and device time.
* The fit (green) follows the per-window minima (black). Their residual is the fit error, and anything above the envelope is network or buffering latency.
* The vendor `Timestamp` column (red) is BrainFlow's running-mean sync, which has no drift model.
* Grey bands mark dropouts.

In [ ]:
display(pd.Series(s.clock.summary()).to_frame("value"))
qp.fig_clock(s)

## 4. Gaps, railing & setup period
Dropouts (data gaps) are listed below and shaded grey in the railing map.

"Railed" means |x| ≥ 90 % of the ADC full scale: ±375 000 µV for EEG (gain 12) and ±1 125 000 µV for EMG/EOG (gain 4).
* Lines: the confirmed valid range (solid green), or, if none, the **suggested** start (dashed grey) and the onset of stable PPG contact.
* The suggestion is a heuristic: the later of 15 s after the last railed sample and the onset of stable PPG contact. Check it in the browser before confirming.
* The bar chart compares railing in the raw data with the GUI's signal-check estimate.

In [ ]:
display(s.gaps if s.gaps is not None and len(s.gaps) else Markdown("No dropouts."))
qp.fig_railing_map(s).show()
qp.fig_railing_bars(s)

## 5. Browser & valid range
* Pick channels per group with Ctrl- or Shift-click; presets are in the dropdown.
* Zoom with box-zoom (double-click resets). Zooming re-fetches full-resolution data. Grey bands mark dropouts; with **blank railed samples** on, railed samples are hidden.
* **Valid range** row:
  * **view suggestion** zooms to the suggested range (dashed green outline), or to the confirmed one.
  * Adjust the view (zoom, or *start / window* + **go**), then **mark view as valid**. It is saved to `galea_session.json` in the session folder; solid green edges mark it, and the excluded part is tinted.
  * **clear** removes it (undefined again).
* Sections 6 and 9 update when you mark or clear the range. Re-run the notebook to update sections 1, 2, 4, 7 and 8.

In [ ]:
b = SignalBrowser(s)
b

## 6. Channel quality
Per ExG channel, over the confirmed valid range (or the whole recording if none is confirmed):
* **rms_1_40_uV:** 1–40 Hz RMS.
* **line_ratio:** 60 Hz PSD / 40–56 Hz PSD.
* **kurtosis:** excess kurtosis of the 1–40 Hz signal, which flags spikes and pops.
* **drift_uV:** 0.1–1 Hz SD.
* **grade:** a heuristic, relative within each group (see `galea.qc.channel_quality`).

In [ ]:
out_quality = W.Output()
cq = None

def show_quality(*_):
    global cq
    t0, t1, label = analysis_range(s)
    cq = channel_quality(s, t0, t1)
    out_quality.clear_output(wait=True)
    with out_quality:
        display(Markdown(f"**Range:** t = {t0:.1f}–{t1:.1f} s ({label})."))
        display(qp.style_grades(cq))
        display(qp.fig_psd(s, t0, t1))

show_quality()
b.on_valid_change(show_quality)
out_quality

## 7. Aux sensors
EDA, PPG and the raw IMU norms, all in file units (the IMU scales in the files are not physical).
* Movement during headset setup shows up in the accelerometer and gyro.
* PPG contact appears as a level jump.

In [ ]:
f = qp.fig_aux(s)
f if f is not None else Markdown("No Aux file.")

In [ ]:
c = qp.constant_channels(s)
Markdown("Constant Aux channels: none.") if c.empty else c

## 8. GUI signal-check
These are the OpenBCI GUI's live µVrms values, as logged. The GUI repeats rows, which are de-duplicated on load. Compare them with section 6.

In [ ]:
f = qp.fig_signalcheck(s)
f if f is not None else Markdown("No signal-check file.")

## 9. Summary

In [ ]:
out_summary = W.Output()

def show_summary(*_):
    out_summary.clear_output(wait=True)
    with out_summary:
        display(Markdown(qp.recommendations(s, q, cq)))

show_summary()
b.on_valid_change(show_summary)
out_summary